---

# Hydrological Response Modeling: A Time Series Analysis of Groundwater Levels in Columbus, Ohio

**Author:** Iaroslav Grushetskyi 

---

---
## Abstract

This study investigates groundwater-level dynamics and forecasting in Columbus, Ohio, using historical groundwater and climatic observations from 2010 to 2015. A time-series modeling framework is developed to characterize temporal dependence, seasonal variability, and the persistence of groundwater levels over time. Autoregressive Integrated Moving Average (ARIMA) and Seasonal ARIMA (SARIMA) models are evaluated to capture non-seasonal and seasonal temporal patterns, respectively, while Long Short-Term Memory (LSTM) neural networks are considered for modeling nonlinear and long-range temporal dependencies. Model performance is evaluated using out-of-sample forecasting metrics, including Mean Absolute Error (MAE), Root Mean Square Error (RMSE), and Mean Absolute Percentage Error (MAPE). The analysis focuses on identifying the temporal structure of groundwater-level fluctuations and determining which forecasting approach most effectively represents the observed dynamics. The results provide a time-series-based baseline for groundwater-level prediction in Columbus, Ohio, supporting improved understanding of aquifer behavior and informing localized groundwater-resource management under changing climatic conditions.

---

---
## Contents
1. [1. Introduction](#1-introduction)

2. [2. Methodology](#2-methodology)  
   - [2.1 Data Acquisition](#21-data-acquisition)  
   - [2.2 Climatological Patching](#22-climatological-patching)  
   - [2.3 Feature Engineering](#23-feature-engineering)  
   - [2.4 Visualizing Trends](#24-visualizing-trends)  

   - [2.5 Supervised Model: Random Forest Regressor](#25-supervised-model-random-forest-regressor)  
     - [2.5.1 Feature Definition](#251-feature-definition)  
     - [2.5.2 Train-Test Split](#252-train-test-split)  
     - [2.5.3 Model Training](#253-model-training)  
     - [2.5.4 Visual Comparison](#254-visual-comparison)  
     - [2.5.5 Model Comparison](#255-model-comparison)  
     - [2.5.6 Detrended Model](#256-detrended-model)  
     - [2.5.7 Summary](#257-summary)  

   - [2.6 Unsupervised Model: K-Means Clustering](#26-unsupervised-model-k-means-clustering)  
     - [2.6.1 Hydro State Features](#261-hydro-state-features)  
     - [2.6.2 Data Preparation](#262-data-preparation)  
     - [2.6.3 Scaling](#263-scaling)  
     - [2.6.4 Elbow Method](#264-elbow-method)  
     - [2.6.5 Model](#265-model)  
     - [2.6.6 Cluster Centroids](#266-cluster-centroids)  
     - [2.6.7 Logic-Based Mapping](#267-logic-based-mapping)  
     - [2.6.8 PCA](#268-pca)  
     - [2.6.9 PCA Loadings](#269-pca-loadings)  
     - [2.6.10 PCA Visualization](#2610-pca-visualization)  
     - [2.6.11 K-Means Summary](#2611-k-means-summary)  

3. [3. Results in the Context of RCPs](#3-results-in-the-context-of-rcps)  

4. [4. Limitations](#4-limitations)  

5. [5. Conclusion](#5-conclusion)  

6. [6. References](#6-references)

---

---
## 1. Introduction
The stability of groundwater resources is increasingly threatened by shifting climatic patterns. In the Midwestern United States, specifically the Scioto River Basin surrounding Columbus, Ohio, the interaction between surface precipitation and subsurface storage is a critical factor for agricultural and municipal planning. 

This project employs machine learning to bridge the gap between atmospheric observations and hydrological responses. By analyzing five years of high-frequency sensor data, we aim to identify the specific climate drivers—such as cumulative precipitation and thermal evapotranspiration—that dictate groundwater fluctuations. This research aligns with the broader framework of the **Representative Concentration Pathways (RCPs)**, as understanding historical climate sensitivity is the prerequisite for projecting how these resources will behave under future warming scenarios.

---

---
## 2. Data Acquisition & Preprocessing
Groundwater and climate data was retrieved from the **USGS National Water Information System (NWIS)** in Columbus, OH, covering the period from **January 1, 2006** to **December 31, 2025**.

---

In [1]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================
import pandas as pd, numpy as np, seaborn as sns
import matplotlib.pyplot as plt

In [2]:
# ============================================================
# 2. GLOBAL SHARPNESS & ACADEMIC STYLE
# ============================================================ 
plt.rcParams.update({ "figure.dpi": 250,
                      "pdf.fonttype": 42,
                      "font.family": "serif", 
                      "axes.labelsize": 11, 
                      "axes.titlesize": 12, 
                      "axes.titleweight": "bold", 
                      "axes.spines.top": False,
                      "axes.spines.right": False, 
                      "axes.grid": True,
                      "grid.alpha": 0.3, 
                      "grid.linestyle": "--", 
                      "legend.fontsize": 9, 
                      "legend.frameon": True, 
                      "legend.edgecolor": "0.8", 
                      "xtick.labelsize": 10, 
                      "ytick.labelsize": 10, 
                      "lines.linewidth": 1.5, 
                      "savefig.bbox": "tight",
                      "savefig.format": "pdf"})
# Set a professional color palette
sns.set_palette("colorblind")

In [14]:
# ============================================================
# 3. FILE PATHS
# ============================================================
water_file = "/Users/Katia/Desktop/Predicting-Groundwater-Levels-Time-Series-Analysis/daily_groundwater_levels.csv"
climate_file = "/Users/Katia/Desktop/Predicting-Groundwater-Levels-Time-Series-Analysis/precipitation_airtemp_data.csv"

In [17]:
# ============================================================
# 4. LOAD THE TWO DATASETS
# ============================================================

water = pd.read_csv(water_file, skiprows = 1)
climate = pd.read_csv(climate_file, skiprows = 1)


# Display basic information
print("Water dataset - first 5 rows:")
print(water.head())
print(water.shape)
print()

print("Climate dataset - first 5 rows:")
print(climate.head())
print(climate.shape)


print("Water dataset - last 5 rows:")
print(water.tail())


print("Climate dataset - last 5 rows:")
print(climate.tail())


Water dataset - first 5 rows:
         date  water_level
0  2006-01-01        21.52
1  2006-01-02        21.38
2  2006-01-03        21.32
3  2006-01-04        21.29
4  2006-01-05        21.28
(7277, 2)

Climate dataset - first 5 rows:
         date  prcp   tmax   tmin
0  2006-01-01   0.00     46    32
1  2006-01-02   0.32     53    44
2  2006-01-03   0.06     51    42
3  2006-01-04   0.00     52    41
4  2006-01-05   0.02     41    34
(7305, 4)
Water dataset - last 5 rows:
            date  water_level
7272  2025-12-27        21.97
7273  2025-12-28        21.90
7274  2025-12-29        21.99
7275  2025-12-30        22.01
7276  2025-12-31        21.87
Climate dataset - last 5 rows:
            date  prcp   tmax   tmin
7300  2025-12-27   0.00     45    39
7301  2025-12-28   0.25     68    40
7302  2025-12-29   0.11     59    20
7303  2025-12-30   0.01     24    20
7304  2025-12-31   0.08     35    21


In [21]:
# Remove leading/trailing spaces and convert names to lowercase
water.columns = water.columns.str.strip().str.lower()
climate.columns = climate.columns.str.strip().str.lower()

print("\nWater-level columns:")
print(water.columns.tolist())

print("\nClimate columns:")
print(climate.columns.tolist())


Water-level columns:
['date', 'water_level']

Climate columns:
['date', 'prcp', 'tmax', 'tmin']


In [22]:
print('The counts of missing values of the following features are:')
print('- precipitation:', climate['prcp'].isna().sum())
print('- maximum temperature:', climate['tmax'].isna().sum())
print('- minimum temperature:', climate['tmin'].isna().sum())
print('- groundwater level:', water['water_level'].isna().sum())

The counts of missing values of the following features are:
- precipitation: 0
- maximum temperature: 0
- minimum temperature: 0
- groundwater level: 0


In [28]:
print('The missing value counts for all the features show zero!')
print('However, the climate dataset has', climate.shape[0], 'rows, and water dataset has,', water.shape[0], 'rows!')
print('This can only mean there are no NaN values in the water data itself, but there are 28 missing dates relative to the complete daily calendar.')
print('Quickly checking the math: 7305 - 7277 rows = ', climate.shape[0]- water.shape[0])

The missing value counts for all the features show zero!
However, the climate dataset has 7305 rows, and water dataset has, 7277 rows!
This can only mean there are no NaN values in the water data itself, but there are 28 missing dates relative to the complete daily calendar.
Quickly checking the math: 7305 - 7277 rows =  28
